# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [1]:
import pandas as pd
# read the sentence data 
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
# check the head
df.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,1,I was rushed into the [PRESSION] two weeks ago...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",2,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",4,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",5,"I got very nauseous, vomiting, dizzy and rejuv..."


In [2]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

In [3]:
# print the size of the data
len(sentences)
# There are 41143 sentences 

41143

### 1.2. Import the list of stopwords

In [4]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/persistent/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load the embedding model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# run the following for embeddings if no pre-saved embeddings can be loaded
#embeddings = embedding_model.encode(sentences, show_progress_bar=True)
# save the pre-calculated embeddings 
#np.save('/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy', embeddings)

/workspace/persistent/mijnidbcoachnlp/venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
# load pre-saved embeddings if you have, otherwise calculate them using the commented-out codes
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
embeddings = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy")

In [7]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

In [8]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Fine-tune the ST model

In [15]:
import random
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Precompute embeddings (this part depends on your chosen embedding model)
# Example: Assuming 'embeddings' is already computed and available
# embeddings = embedding_model.encode(sentences)

# Initialize BERTopic with the pre-fit vectorizer
topic_model = BERTopic()

# Define parameter ranges
range_n_neighbors = [10, 20, 30, 40, 50]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.05, 0.1]

# Number of random configurations to generate
num_configs = 3  # Adjust this as needed

# Generate random UMAP configurations
umap_configs = [
    {
        "n_neighbors": random.choice(range_n_neighbors),
        "n_components": random.choice(range_n_components),
        "min_dist": random.choice(range_min_dist),
    }
    for _ in range(num_configs)
]

# Define the ranges for HDBSCAN configurations
range_min_cluster_size = [10, 20, 30]  # Example values
range_min_samples = [10, 20, 30]  # Example values

# Generate random HDBSCAN configurations
hdbscan_configs = [
    {
        "min_cluster_size": random.choice(range_min_cluster_size),
        "min_samples": random.choice([x for x in range_min_samples if x <= random.choice(range_min_cluster_size)]),
    }
    for _ in range(num_configs)
]

# Initialize a dictionary to keep track of configurations and their associated coherence scores
tried_configs = {}

# Optionally, load previously tried configurations from a file (if running multiple times)
try:
    with open("tried_configs_st.pkl", "rb") as f:
        tried_configs = pickle.load(f)
except FileNotFoundError:
    pass  # If no previous file exists, we start fresh

# Variables to track the best model
best_topic_model = None
best_coherence_score = -float('inf')  # Initializing to a very low score
best_configure = None

# Set the outlier thresholds (upper and lower)
upper_outlier_threshold = 0.5  # Maximum allowed outlier percentage (5% of points)
lower_outlier_threshold = 0.2  # Minimum allowed outlier percentage (2% of points)

In [17]:
# Fine-tuning loop
for umap_params in umap_configs:
    for hdbscan_params in hdbscan_configs:
        config_tuple = (frozenset(umap_params.items()), frozenset(hdbscan_params.items()))

        # Skip if the configuration has already been tried
        if config_tuple in tried_configs:
            print(f"Skipping already tried configuration: {umap_params}, {hdbscan_params}")
            continue

        # Initialize UMAP and HDBSCAN with current configurations
        umap_model = UMAP(**umap_params)
        hdbscan_model = HDBSCAN(**hdbscan_params)
        print(f"Fitting model for UMAP {umap_params} and HDBSCAN {hdbscan_params}")

        # Update BERTopic with the new UMAP and HDBSCAN models
        topic_model.umap_model = umap_model
        topic_model.hdbscan_model = hdbscan_model
        topic_model.vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')
        topic_model.calculate_probabilities = False

        # Fit the model
        topics, probs = topic_model.fit_transform(sentences, embeddings)

        # Calculate coherence score
        current_coherence_score = calculate_c_v(topic_model)

        # Check if the model has at least 20 topics
        num_topics = len(topic_model.get_topics())
        if num_topics < 20:
            print(f"Skipping model with {num_topics} topics (less than 20).")
            tried_configs[config_tuple] = "skipped"
            continue

        # Check outlier percentage
        outlier_cluster_size = sum(1 for label in hdbscan_model.labels_ if label == -1)
        outlier_percentage = outlier_cluster_size / len(hdbscan_model.labels_)
        
        if outlier_percentage > upper_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (above {upper_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        elif outlier_percentage < lower_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (below {lower_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        
        # Store the valid configuration and score
        tried_configs[config_tuple] = current_coherence_score

        # Update best model
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = topic_model
            best_configure = (umap_params, hdbscan_params)

        print(f"Coherence score: {current_coherence_score}\n")

        # Save tried configurations after every iteration
        with open("tried_configs_st.pkl", "wb") as f:
            pickle.dump(tried_configs, f)


Fitting model for UMAP {'n_neighbors': 50, 'n_components': 4, 'min_dist': 0.05} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 10}
Coherence score: 0.3662643130528934

Fitting model for UMAP {'n_neighbors': 50, 'n_components': 4, 'min_dist': 0.05} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 20}
Coherence score: 0.35050085496611455

Fitting model for UMAP {'n_neighbors': 50, 'n_components': 4, 'min_dist': 0.05} and HDBSCAN {'min_cluster_size': 30, 'min_samples': 10}
Coherence score: 0.3582903721014854

Fitting model for UMAP {'n_neighbors': 20, 'n_components': 4, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 10}
Coherence score: 0.3666985394181407

Fitting model for UMAP {'n_neighbors': 20, 'n_components': 4, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 20}
Coherence score: 0.36657597532332886

Fitting model for UMAP {'n_neighbors': 20, 'n_components': 4, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 30, 'min_samples': 10}


In [18]:
# After the loop, print the best configurations and score
print("Best UMAP and HDBSCAN configurations based on coherence score:")
print(f"Best Coherence Score: {best_coherence_score}")
print(f"Best UMAP Params: {best_configure[0]}")
print(f"Best HDBSCAN Params: {best_configure[1]}")

Best UMAP and HDBSCAN configurations based on coherence score:
Best Coherence Score: 0.368752327810459
Best UMAP Params: {'n_neighbors': 20, 'n_components': 3, 'min_dist': 0.05}
Best HDBSCAN Params: {'min_cluster_size': 10, 'min_samples': 10}


In [20]:
import pickle

# Load the file
with open("tried_configs_st.pkl", "rb") as f:
    tried_configs = pickle.load(f)

# Print the contents
for config, score in tried_configs.items():
    print(f"Config: {config}, Score: {score}")


Config: (frozenset({('n_neighbors', 10), ('min_dist', 0.1), ('n_components', 5)}), frozenset({('min_cluster_size', 30), ('min_samples', 10)})), Score: 0.3425443099837642
Config: (frozenset({('n_neighbors', 10), ('n_components', 4), ('min_dist', 0.0)}), frozenset({('min_samples', 30), ('min_cluster_size', 20)})), Score: 0.35067638061555395
Config: (frozenset({('n_neighbors', 10), ('n_components', 4), ('min_dist', 0.0)}), frozenset({('min_cluster_size', 30), ('min_samples', 10)})), Score: 0.3362990734428644
Config: (frozenset({('n_neighbors', 10), ('n_components', 4), ('min_dist', 0.0)}), frozenset({('min_samples', 30), ('min_cluster_size', 10)})), Score: 0.3490907784930923
Config: (frozenset({('n_neighbors', 10), ('n_components', 3), ('min_dist', 0.0)}), frozenset({('min_samples', 30), ('min_cluster_size', 20)})), Score: 0.33991947203180106
Config: (frozenset({('n_neighbors', 10), ('n_components', 3), ('min_dist', 0.0)}), frozenset({('min_cluster_size', 30), ('min_samples', 10)})), Scor

In [23]:
best_topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [24]:
calculate_c_v(best_topic_model)

0.3552550163346731

In [21]:
best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18597,-1_afspraak_vraag_gaat_weet,"[afspraak, vraag, gaat, weet, volgende, goed, ...","[en/of , Het gaat de laatste 2 weken niet zo g..."
1,0,3367,0_bloed_prikken_bloedprikken_ontlasting,"[bloed, prikken, bloedprikken, ontlasting, sli...",[Afgelopen vrijdag ben ik bloed laten prikken....
2,1,1291,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...","[Ja bij mijn huisarts., Ja bij mijn huisarts.,..."
3,2,556,2_medicatie_apotheek_medicijnen_medicijn,"[medicatie, apotheek, medicijnen, medicijn, re...","[Nog steeds geen recept binnen bij apotheek!, ..."
4,3,516,3_pijn_buikpijn_pijnlijk_buik,"[pijn, buikpijn, pijnlijk, buik, pijnlijke, ru...","[Dit is ondoenlijk van wege de pijn!, Maar ik ..."
...,...,...,...,...,...
242,241,11,241_doorgestuurd_apotheekderrein_zorginstellin...,"[doorgestuurd, apotheekderrein, zorginstelling...",[Het recept van de picoprep zou zijn doorgestu...
243,242,11,242_optie_rele_alternatieve_buitenland,"[optie, rele, alternatieve, buitenland, vastge...","[Zijn er nog andere optie?, Is dit een reële o..."
244,243,10,243_verwachtte_kim_beloofd_schrijft,"[verwachtte, kim, beloofd, schrijft, weekend, ...",[Ik had beloofd nog even te laten weten hoe he...
245,244,10,244_contactmoment_emailen_vlg_simpony,"[contactmoment, emailen, vlg, simpony, kontakt...",[ik heb vorige week via de mail nieuwe simpon...


In [25]:
# check if reduce outliers now work
# Reduce outliers
topics = best_topic_model.topics_
new_topics = best_topic_model.reduce_outliers(sentences, topics)

In [26]:
best_topic_model.update_topics(sentences, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'))

2025-02-13 05:22:51,663 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [27]:
# check topic_info after outlier reduction
best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_heermevrouw_aangenaam,"[verstandigst, trant, heermevrouw, aangenaam, ...","[en/of , Het gaat de laatste 2 weken niet zo g..."
1,0,3567,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, sli...",[Afgelopen vrijdag ben ik bloed laten prikken....
2,1,1458,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...","[Ja bij mijn huisarts., Ja bij mijn huisarts.,..."
3,2,650,2_medicatie_medicijn_apotheek_medicijnen,"[medicatie, medicijn, apotheek, medicijnen, vo...","[Nog steeds geen recept binnen bij apotheek!, ..."
4,3,694,3_pijn_buikpijn_pijnlijk_buik,"[pijn, buikpijn, pijnlijk, buik, pijnlijke, ru...","[Dit is ondoenlijk van wege de pijn!, Maar ik ..."
...,...,...,...,...,...
242,241,36,241_doorgestuurd_picoprep_gestuurd_doorgezet,"[doorgestuurd, picoprep, gestuurd, doorgezet, ...",[Het recept van de picoprep zou zijn doorgestu...
243,242,75,242_optie_alternatief_momenten_mogelijkheden,"[optie, alternatief, momenten, mogelijkheden, ...","[Zijn er nog andere optie?, Is dit een reële o..."
244,243,61,243_nou_wekelijks_geplande_volgen,"[nou, wekelijks, geplande, volgen, geval, inna...",[Ik had beloofd nog even te laten weten hoe he...
245,244,48,244_gelezen_laboratorium_buikgriep_afgesproken,"[gelezen, laboratorium, buikgriep, afgesproken...",[ik heb vorige week via de mail nieuwe simpon...


In [28]:
best_topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model_ro", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [29]:
calculate_c_v(best_topic_model)

0.3566552534427884

# map the topics to one of the two labels

In [173]:
from bertopic import BERTopic
topic_model = BERTopic.load("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model_ro", embedding_model=embedding_model)

/workspace/persistent/mijnidbcoachnlp/venv/lib/python3.8/site-packages/bertopic/_save_utils.py:187: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

/workspace/persistent/mijni

In [24]:
topic_model.get_topic_info().head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_heermevrouw_aangenaam,"[verstandigst, trant, heermevrouw, aangenaam, ...",NaN
1,0,3567,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, sli...",NaN
2,1,1458,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...",NaN
3,2,650,2_medicatie_medicijn_apotheek_medicijnen,"[medicatie, medicijn, apotheek, medicijnen, vo...",NaN
4,3,694,3_pijn_buikpijn_pijnlijk_buik,"[pijn, buikpijn, pijnlijk, buik, pijnlijke, ru...",NaN


In [174]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 245/245 [00:00<00:00, 354.97it/s]


In [175]:
# Reduce the topics with a similarity threshold < 0.5
threshold = 0.5 # Define the threshold for merging
filtered_hierarchical_topics = hierarchical_topics[hierarchical_topics["Distance"] <= threshold]

# Ensure the loop stops when no more topics can be merged
while not filtered_hierarchical_topics.empty:  

    list_topics_to_merge = filtered_hierarchical_topics["Topics"].to_list()

    # Merge topics
    topic_model.merge_topics(sentences, list_topics_to_merge)

    # Recompute hierarchical topics after merging
    hierarchical_topics = topic_model.hierarchical_topics(sentences)

    # Update the filtered topics for the next iteration
    filtered_hierarchical_topics = hierarchical_topics[hierarchical_topics["Distance"] <= threshold]


100%|██████████| 208/208 [00:00<00:00, 361.62it/s]


In [110]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 208/208 [00:00<00:00, 358.05it/s]


In [176]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, onderwerpen, genegeerd, docter, dexamfetamine, glucose, hogen]","[Of iets in die trant., Wat is voor mij het verstandigst., Of iets in die trant.]"
1,0,3567,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, slijm, laten, bloeduitslagen, ingeleverd, geprikt, inleveren]","[Maar moet ik niet ook nog bloed laten prikken?, ik nooit meer bloed te laten prikken?, Afgelopen maandag 4 juli heb ik bloed laten prikken via .]"
2,1,1458,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, medische, verpleegkundige, behandelend, behandeling]","[Volgens de huisarts heb ik ze gehad in '89., huisarts heeft ca., In het ziekenhuis of de huisarts.]"
3,2,1049,2_recept_nieuw_apotheek_sturen,"[recept, nieuw, apotheek, sturen, herhaalrecept, purinethol, nethol, aanvragen, puri, uitschrijven]","[Zou u een nieuw recept naar de apotheek in het [ZIEKENHUIS] kunnen sturen?, Zouden jullie een nieuw recept naar mijn apotheek in willen sturen?, Zou u voor mij een nieuw recept naar mijn apothee..."
4,3,819,3_pijn_buikpijn_buik_pijnlijk,"[pijn, buikpijn, buik, pijnlijk, rug, pijnlijke, onderbuik, gewrichten, zeurende, erger]","[Dit is ondoenlijk van wege de pijn!, Pijn is al wat minder., De dag daarna geen pijn.]"
...,...,...,...,...,...
205,204,35,204_eerlijk_geboorte_gezegd_hoor,"[eerlijk, geboorte, gezegd, hoor, tips, graag, animo, periodecheck, oudste, verschil]","[Ik neem aan dat ze daar eerlijk over is., Ik merk eerlijk gezegd geen verschil., Erg prettig voel ik me er niet bij, eerlijk gezegd... Ik hoor graag van je,]"
206,205,34,205_geboortedatum_geboorte_geb_geboren,"[geboortedatum, geboorte, geb, geboren, hpm, afschaffen, inlevering, welkom, datums, woog]","[De geboortedatum is ., geboortedatum is: ., Mijn geboortedatum is .]"
207,206,31,206_nieuws_goed_goede_geloven,"[nieuws, goed, goede, geloven, super, verwacht, fantastisch, slecht, eraan, geannuleerd]","[dat is goed nieuws., Dat is heel goed nieuws!, Nu ga ik er vanuit dat geen nieuws, goed nieuws is.]"
208,207,30,207_ggd_hoor_tel_graag,"[ggd, hoor, tel, graag, eem, secretaresse, uitnodigingen, gebruikers, zsm, doelgroep]","[Graag hoor ik van u, Graag hoor ik van jullie, Graag horen we vanuit jullie of het klopt wat de GGD zei]"


In [72]:
calculate_c_v(topic_model)

0.3686722770522602

In [177]:
topics_to_merge_manually = [[190, 25],#food, appetite
[65, 3],# GI symptoms
[108, 54], #sleep
[171, 7, 89, 83, 57, 115, 60, 5, 93], #medicine/medication use
[92, 90, 2], #pharmacy and prescription
[193, 107, 173, 70], # telephone contact
[180, 87, 151], # vitamins and minerals
[197, 59], #inquire of test results
[80, 72], # advice
[113, 156, 18], # coronavirus & vaccine
[124, 142], #fever
[11, 130], # quetion/questionaire
[42, 94, 14, 183], #calprotectin home-test
[189, 20, 149, 162, 55, 102, 167, 208, 154, 53, 109, 187, 98, 78, 52, 101, 23, 22, 179] # sign-offs and greetings
]
topic_model.merge_topics(sentences, topics_to_merge_manually)


In [121]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, ...","[Of iets in die trant., Wat is voor mij het ve..."
1,0,3567,0_bloed_prikken_ontlasting_slijm,"[bloed, prikken, ontlasting, slijm, bloedprikk...",[Afgelopen maandag 4 juli heb ik bloed laten p...
2,1,2884,1_dank_bedankt_reactie_snelle,"[dank, bedankt, reactie, snelle, alvast, beric...","[Dank voor jullie snelle reactie., Dank voor u..."
3,2,2298,2_medicatie_gebruik_tabletten_daags,"[medicatie, gebruik, tabletten, daags, medicij...","[ik sik nu 1 x daags 3 tabletten van 3 mg., he..."
4,3,1458,3_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...","[Of moet het via de huisarts?, De huisarts vro..."
...,...,...,...,...,...
160,159,36,159_doorgestuurd_picoprep_gestuurd_zorginstelling,"[doorgestuurd, picoprep, gestuurd, zorginstell...",[Dit mag doorgestuurd worden naar de [APOTHEEK...
161,160,35,160_eerlijk_hoor_geboorte_gezegd,"[eerlijk, hoor, geboorte, gezegd, graag, tips,...","[Ik neem aan dat ze daar eerlijk over is., Ik ..."
162,161,34,161_geboortedatum_geboorte_geb_geboren,"[geboortedatum, geboorte, geb, geboren, hpm, a...","[geboortedatum is: ., geboortedatum is: ., Mij..."
163,162,31,162_nieuws_goed_goede_geloven,"[nieuws, goed, goede, geloven, super, verwacht...","[goed nieuws dus., Dat is goed nieuws., Nu ga ..."


In [122]:
topic_names = topic_model.get_topic_info()["Name"].to_list()
topic_names

['-1_verstandigst_trant_aangenaam_zenuwachtig',
 '0_bloed_prikken_ontlasting_slijm',
 '1_dank_bedankt_reactie_snelle',
 '2_medicatie_gebruik_tabletten_daags',
 '3_huisarts_arts_dokter_mdl',
 '4_recept_apotheek_nieuw_sturen',
 '5_pijn_buikpijn_buik_pijnlijk',
 '6_thuis_cal_quanton_vorige',
 '7_hoor_graag_horen_alvast',
 '8_merk_last_tijd_laatste',
 '9_vraag_vragen_vraagje_vragenlijst',
 '10_voel_gevoel_voelt_voelen',
 '11_corona_vaccin_vaccinatie_positief',
 '12_contact_telefonisch_opnemen_consult',
 '13_gepland_staan_poli_afspraak',
 '14_wacht_hoop_wachten_hopelijk',
 '15_telefoon_bereikbaar_nummer_oproep',
 '16_klachten_kleine_vermoeidheid_dun',
 '17_goed_gaat_ging_algemeen',
 '18_test_gedaan_zelftest_testen',
 '19_eten_eet_eetlust_drinken',
 '20_sessie_uitnodiging_afspraak_nieuwe',
 '21_advies_adviseren_overleggen_overleg',
 '22_nacht_wakker_slaap_vannacht',
 '23_toilet_diarree_vaker_aandrang',
 '24_infuus_komende_echo_vanmiddag',
 '25_weten_laat_laten_duidelijkheid',
 '26_afspraak_v

In [178]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 163/163 [00:00<00:00, 363.05it/s]


In [143]:
doc_info = topic_model.get_document_info(sentences)
pd.set_option("display.max_colwidth", None)
doc_info[doc_info["Topic"] == 45]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
456,Ik heb dus morgen een buisje bij me met ontlasting alleen heb ik geen formulier erbij.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
566,Ik zal bij de eerst volgende keer potje ontlasting meenemen.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
629,Ik heb pas een ontlastingformulier ontvangen met een potje.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
662,ik ben weer aan mijn laatste potje purinethol toe.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
733,"Laboratoriumformulieren heb ik niet, potje wel..",45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
...,...,...,...,...,...,...,...
40350,Ook heb ik een formulier gekregen voor de ontlasting.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
40440,"Nee heb geen potje meer, zou u deze kunnen opsturen.",45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
40615,Ik heb geen lab formulieren/buisjes meer.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potjes - lab - buisje - ontvangen - bezit - prikformulieren - informatiefolder,False
40623,Ik heb het formulier en potje nog niet ontvangen.,45,45_potje_formulier_formulieren_potjes,"[potje, formulier, formulieren, potjes, lab, buisje, ontvangen, bezit, prikformulieren, informatiefolder]","[Ik heb nog een formulier en potje., Formulier en potje heb ik., Ja ik heb nog een potje en een formulier.]",potje - formulier - formulieren - potj

In [179]:
topics_to_merge_manually_2 = [[5, 131], # GI symptoms
[16, 8], # general symptoms
[34, 23], # diarrhea
[101, 3], # hospital/doctor visit
[20, 26, 13, 50], # appointment/session
[88, 95, 74], # calprotectin value
[18, 6], # home test
[45, 37], # stool sampling forms and papers
[29, 89, 134], # mail and address
[162, 67, 1], # sign-offs
[56, 104], # non-actionable 
]
topic_model.merge_topics(sentences, topics_to_merge_manually_2)


In [146]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, onderwerpen, genegeerd, docter, dexamfetamine, glucose, hogen]","[Of iets in die trant., Wat is voor mij het verstandigst., Of iets in die trant.]"
1,0,3567,0_bloed_prikken_ontlasting_slijm,"[bloed, prikken, ontlasting, slijm, laten, bloedprikken, bloeduitslagen, ingeleverd, geprikt, inleveren]","[Maar moet ik niet ook nog bloed laten prikken?, ik nooit meer bloed te laten prikken?, Afgelopen maandag 4 juli heb ik bloed laten prikken via .]"
2,1,3093,1_bedankt_dank_reactie_snelle,"[bedankt, dank, reactie, snelle, alvast, bericht, antwoord, dankjewel, fijn, weekend]","[dank voor jouw snelle reactie!, Alvast dank voor jullie reactie., dank voor de snelle reactie, ja dat zou fijn zijn.]"
3,2,2298,2_medicatie_gebruik_tabletten_medicijnen,"[medicatie, gebruik, tabletten, medicijnen, daags, gebruiken, pentasa, tablet, medicijn, apotheek]","[Op dit moment heb ik nog voor één week klysma's en 150 stuks Pentasa (2 maal daags 2 tabletten = 37 dg)., heer, mevrouw, Graag had ik een herhaal recept voor: Puri-Nethol 50 mg tabletten (mercaptopurinemonohydraat 50 mg), 1x daags 1 tablet., Deze medicijn kon ik gebruiken samen met mijn andere medicatie.]"
4,3,1559,3_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, verpleegkundige, medische, afspraak, behandelend]","[Kan dit alles bij de huisarts?, huisarts heeft ca., Ik ben net bij de huisarts geweest.]"
...,...,...,...,...,...
144,143,37,143_terugkoppeling_terugkoppelen_bedankt_dank,"[terugkoppeling, terugkoppelen, bedankt, dank, supersnelle, gekoppeld, bus, ontlastingsmonster, branden, baat]","[voor de terugkoppeling., voor de terugkoppeling., Bij deze dus de terugkoppeling.]"
145,144,36,144_doorgestuurd_picoprep_gestuurd_zorginstelling,"[doorgestuurd, picoprep, gestuurd, zorginstelling, doorgezet, derrein, recept, apotheek, gram, hartbewaking]","[Dit mag doorgestuurd worden naar de [APOTHEEK] in ., Het recept van de picoprep zou zijn doorgestuurd naar [APOTHEEK]derrein ., Dit mag doorgestuurd worden naar de [APOTHEEK] in .]"
146,145,35,145_eerlijk_hoor_gezegd_geboorte,"[eerlijk, hoor, gezegd, geboorte, graag, tips, verschil, animo, periodecheck, oudste]","[Ik neem aan dat ze daar eerlijk over is., Ik merk eerlijk gezegd geen verschil., Erg prettig voel ik me er niet bij, eerlijk gezegd... Ik hoor graag van je,]"
147,146,34,146_geboortedatum_geboorte_geb_geboren,"[geboortedatum, geboorte, geb, geboren, hpm, afschaffen, inlevering, welkom, datums, woog]","[geboortedatum is: ., Met vriendlijke (geboortedatum ), De geboortedatum is .]"


In [149]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 147/147 [00:00<00:00, 366.66it/s]


In [148]:
topic_names = topic_model.get_topic_info()["Name"].to_list()
topic_names

['-1_verstandigst_trant_aangenaam_zenuwachtig',
 '0_bloed_prikken_ontlasting_slijm',
 '1_bedankt_dank_reactie_snelle',
 '2_medicatie_gebruik_tabletten_medicijnen',
 '3_huisarts_arts_dokter_mdl',
 '4_afspraak_gepland_sessie_poli',
 '5_recept_apotheek_nieuw_sturen',
 '6_test_thuis_gedaan_quanton',
 '7_pijn_buikpijn_buik_pijnlijk',
 '8_klachten_merk_last_tijd',
 '9_hoor_graag_horen_alvast',
 '10_toilet_keer_diarree_vaker',
 '11_vraag_vragen_vraagje_vragenlijst',
 '12_voel_gevoel_voelt_voelen',
 '13_corona_vaccinatie_vaccin_positief',
 '14_contact_telefonisch_opnemen_consult',
 '15_formulieren_formulier_potje_papieren',
 '16_mail_brief_adres_bericht',
 '17_wacht_hoop_wachten_hopelijk',
 '18_telefoon_bereikbaar_nummer_oproep',
 '19_goed_gaat_ging_algemeen',
 '20_calprotectine_waarde_waardes_bekend',
 '21_eten_eet_eetlust_drinken',
 '22_advies_adviseren_overleggen_graag',
 '23_nacht_wakker_slaap_vannacht',
 '24_infuus_komende_echo_vanmiddag',
 '25_weten_laat_laten_duidelijkheid',
 '26_crohn_

In [171]:
doc_info = topic_model.get_document_info(sentences)
pd.set_option("display.max_colwidth", 200)
doc_info[doc_info["Topic"] == 57]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
458,Half 10 heb ik de echo.,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
495,"en , Op 6 april om 22:37 uur ben ik bevallen van onze dochter .",57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
1257,Sinds de scopie van 2 november zijn mijn darmklachten verslechterd.,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
1259,Enige pluspuntje is dat ik sinds 7 november 's nachts af en toe scheetjes laat want dat ging vanaf eind oktober niet meer.,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
1267,sinds zaterdag 11 november gebruik ik een verhoogd schema prednisolon.,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
...,...,...,...,...,...,...,...
40430,oktober/november 2021 ben ik op controle geweest bij .,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
40528,In januari heb ik een afspraak opstaan en daarvoor heb ik een lab briefje gekregen.,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
40554,Dus dat zou dan ook eind februari zijn,57,57_telefonische_eind_opstaan_verzet,"[telefonische, eind, opstaan, verzet, entocort, half, bevallen, afspraak, dochter, kom]","[Moet ik zelf een telefonische afspraak maken?, Afgelopen maandag had ik een telefonische afspraak met ., Afgelopen maandag 20-12 zou ik een telefonische afspraak hebben met .]",telefonische - eind - opstaan - verzet - entocort - half - bevallen - afspraak - dochter - kom,False
40810,De laatste medicatie is ent

In [180]:
# large merge
topics_to_merge_manually_3 = [[145, 147, 9, 123], # hoor graag
[25, 68], # want to know
[33, 34], # results inquiry
[102, 42], # weet
[92, 55], # thought process
[143, 1],
[78, 100], # operation
[18, 63, 59, 39, 44, 4, 57] # appointment and contact
]
topic_model.merge_topics(sentences, topics_to_merge_manually_3)


In [182]:
topic_model.get_topic_info()[:10]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, onderwerpen, genegeerd, docter, dexamfetamine, glucose, hogen]","[Of iets in die trant., Wat is voor mij het verstandigst., Of iets in die trant.]"
1,0,3567,0_bloed_prikken_ontlasting_laten,"[bloed, prikken, ontlasting, laten, slijm, bloedprikken, ingeleverd, bloeduitslagen, geprikt, inleveren]","[ik nooit meer bloed te laten prikken?, Afgelopen maandag 4 juli heb ik bloed laten prikken via ., Maar moet ik niet ook nog bloed laten prikken?]"
2,1,3130,1_bedankt_dank_reactie_snelle,"[bedankt, dank, reactie, snelle, alvast, bericht, antwoord, dankjewel, fijn, weekend]","[heer , Oké, dank u voor de snelle reactie!, Alvast dank voor uw reactie, Fijn bedankt voor snelle reactie]"
3,2,2871,2_afspraak_bellen_proberen_bereikbaar,"[afspraak, bellen, proberen, bereikbaar, gepland, poli, vandaag, telefoon, staan, staat]","[Ik zal een afspraak maken., De afspraak is inmiddels op een andere dag vastgezet., Ik heb afgelopen vrijdag een afspraak met gehad.]"
4,3,2298,3_medicatie_gebruik_tabletten_medicijnen,"[medicatie, gebruik, tabletten, medicijnen, daags, gebruiken, pentasa, tablet, apotheek, medicijn]","[Graag zou ik andere medicatie willen., ik zou graag een recept voor 3 maanden willen hebben van : Azathioprine 25 mg 1 x daags 1 tablet Azathioprine 50 mg 1 x daags 2 tablet Forlax 10 gram 4 x da..."
5,4,1559,4_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, verpleegkundige, medische, afspraak, behandeling]","[De huisarts vroeg zich af of dit kan door ., Ik ben afgelopen vrijdag weer bij de huisarts geweest., Ja via de huisarts.]"
6,5,1347,5_recept_apotheek_nieuw_sturen,"[recept, apotheek, nieuw, sturen, herhaalrecept, faxen, purinethol, nethol, puri, aanvragen]","[Kunt u een nieuw recept sturen naar apotheek ?, Kunnen jullie a.u.b. een nieuw recept naar de apotheek sturen., Zou u anders een nieuw recept kunnen sturen naar mijn apotheek?]"
7,6,1239,6_test_thuis_gedaan_quanton,"[test, thuis, gedaan, quanton, cal, testen, vorige, thuistest, huis, zelftest]","[Moet ik deze test nu doen ?, De test was echter nog niet over de datum., Ik heb inderdaad nog een test thuis.]"
8,7,1088,7_pijn_buikpijn_buik_pijnlijk,"[pijn, buikpijn, buik, pijnlijk, last, pijnlijke, rug, gevoel, onderbuik, hoofdpijn]","[Geen pijn meer in., Maar ik heb af en toe meer pijn., Deze doet niet pijn, het groeit wel.]"
9,8,1012,8_klachten_merk_last_tijd,"[klachten, merk, last, tijd, darmen, laatste, vermoeidheid, kleine, geworden, dun]","[ik heb de laatste tijd weer meer last van mijn buik., Ik heb de laatste dagen meer klachten van mijn darmen., Nu merk ik sinds gisteren dat ik meer last heb van mijn darmen.]"


In [183]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 132/132 [00:00<00:00, 364.61it/s]


In [185]:
# large merge
topics_to_merge_manually_4 = [[106, 78, 7, 8, 39, 24, 10, 66], # symptoms and complaints
[0, 6], # stool and blood test
[21, 15, 63], # calprotectin home test
[14, 2], #contact and appointment
]
topic_model.merge_topics(sentences, topics_to_merge_manually_4)


In [191]:

topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, onderwerpen, genegeerd, docter, dexamfetamine, glucose, hogen]","[Of iets in die trant., Wat is voor mij het verstandigst., Of iets in die trant.]"
1,0,4806,0_bloed_prikken_ontlasting_test,"[bloed, prikken, ontlasting, test, laten, slijm, bloedprikken, ingeleverd, bloeduitslagen, geprikt]","[Ik heb echter nog geen bloed laten prikken., Ik heb overigens gisterenochtend bloed laten prikken., m.v.g. Ben op in kerkrade geweest voor bloed te laten prikken.]"
2,1,3651,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee..."
3,2,3416,2_afspraak_telefonisch_contact_bellen,"[afspraak, telefonisch, contact, bellen, proberen, bereikbaar, vandaag, poli, gepland, opnemen]","[Kan een afspraak via u, of neem ik hier zelf contact over op?, 22 november heb ik telefonisch contact gehad met ., Anders moeten we nog even telefonisch contact op nemen]"
4,3,3130,3_bedankt_dank_reactie_snelle,"[bedankt, dank, reactie, snelle, alvast, bericht, antwoord, dankjewel, fijn, weekend]","[Dank u voor de snelle reactie, Alvast dank voor jullie reactie., Bedankt voor uw snelle reactie.dat is heel fijn.]"
...,...,...,...,...,...
118,117,42,117_betekent_vervolgstappen_stappen_the,"[betekent, vervolgstappen, stappen, the, vervolg, vervolgstap, leukocyten, rapportage, attach, far]","[Wat betekent dit nu?, Wat betekent dit?, Wat betekent dit?]"
119,118,39,118_aangepast_baat_beclometason_tabletjes,"[aangepast, baat, beclometason, tabletjes, middel, nachtrust, weerstand, beviel, concluderen, echinacea]","[Zou dit ook nog aangepast kunnen worden?, De gezondheidspremie wordt aangepast., Ik heb hier toch veel meer baat bij dan bij de Beclometason 3x per week en nu gaat het goed met de klachten.]"
120,119,37,119_tegemoet_zie_reactie_graag,"[tegemoet, zie, reactie, graag, hieronder, kliniek, papiertje, premievrijstelling, afsprakenbrief, pensioenfonds]","[Ik zie graag jullie reactie tegemoet., Graag zie ik u reactie tegemoet., Ik zie je reactie graag tegemoet.]"
121,120,36,120_doorgestuurd_picoprep_gestuurd_zorginstelling,"[doorgestuurd, picoprep, gestuurd, zorginstelling, doorgezet, derrein, recept, apotheek, gram, hartbewaking]","[Dit mag doorgestuurd worden naar de [APOTHEEK] in ., Dit mag doorgestuurd worden naar de [APOTHEEK] in ., Het recept van de picoprep zou zijn doorgestuurd naar [APOTHEEK]derrein .]"


In [190]:
topic_info = topic_model.get_topic_info()
topic_names = topic_info["Name"].to_list()
topic_names

['-1_verstandigst_trant_aangenaam_zenuwachtig',
 '0_bloed_prikken_ontlasting_test',
 '1_pijn_buikpijn_toilet_last',
 '2_afspraak_telefonisch_contact_bellen',
 '3_bedankt_dank_reactie_snelle',
 '4_medicatie_gebruik_tabletten_medicijnen',
 '5_huisarts_arts_dokter_mdl',
 '6_recept_apotheek_nieuw_sturen',
 '7_bekend_calprotectine_uitslag_waarde',
 '8_hoor_graag_horen_verneem',
 '9_vraag_vragen_vraagje_vragenlijst',
 '10_voel_gevoel_voelt_voelen',
 '11_corona_vaccinatie_vaccin_positief',
 '12_formulieren_formulier_potje_papieren',
 '13_mail_brief_adres_bericht',
 '14_weten_laat_vertellen_laten',
 '15_hoop_wacht_wachten_hopelijk',
 '16_goed_gaat_ging_algemeen',
 '17_eten_eet_eetlust_drinken',
 '18_advies_adviseren_overleggen_graag',
 '19_infuus_komende_echo_vanmiddag',
 '20_weet_ermee_goed_houdt',
 '21_crohn_ziekte_verklaring_humira',
 '22_prima_helemaal_allemaal_zsm',
 '23_beter_gebeurd_gaat_stuk',
 '24_ijzer_vitamine_ijzerinfuus_foliumzuur',
 '25_maanden_maand_vragenlijst_maandelijks',
 '2

In [238]:
labels = [
    "A",  # -1_verstandigst_trant_aangenaam_zenuwachtig
    "M",  # 0_bloed_prikken_ontlasting_test
    "M",  # 1_pijn_buikpijn_toilet_last
    "A",  # 2_afspraak_telefonisch_contact_bellen
    "A",  # 3_bedankt_dank_reactie_snelle
    "M",  # 4_medicatie_gebruik_tabletten_medicijnen
    "M",  # 5_huisarts_arts_dokter_mdl
    "A",  # 6_recept_apotheek_nieuw_sturen
    "M",  # 7_bekend_calprotectine_uitslag_waarde
    "A",  # 8_hoor_graag_horen_verneem
    "A",  # 9_vraag_vragen_vraagje_vragenlijst
    "M",  # 10_voel_gevoel_voelt_voelen
    "M",  # 11_corona_vaccinatie_vaccin_positief
    "A",  # 12_formulieren_formulier_potje_papieren
    "A",  # 13_mail_brief_adres_bericht
    "A",  # 14_weten_laat_vertellen_laten
    "A",  # 15_hoop_wacht_wachten_hopelijk
    "M",  # 16_goed_gaat_ging_algemeen
    "M",  # 17_eten_eet_eetlust_drinken
    "M",  # 18_advies_adviseren_overleggen_graag
    "M",  # 19_infuus_komende_echo_vanmiddag
    "A",  # 20_weet_ermee_goed_houdt
    "M",  # 21_crohn_ziekte_verklaring_humira
    "A",  # 22_prima_helemaal_allemaal_zsm
    "M",  # 23_beter_gebeurd_gaat_stuk
    "M",  # 24_ijzer_vitamine_ijzerinfuus_foliumzuur
    "A",  # 25_maanden_maand_vragenlijst_maandelijks
    "A",  # 26_mogelijk_mogelijkheid_manier_snel
    "M",  # 27_stoppen_rekening_cortiment_doorgaan
    "A",  # 28_idee_denk_dacht_meld
    "A",  # 29_middag_zetten_aanstaande_morgen
    "A",  # 30_geregeld_opsturen_gekeken_doorgeven
    "M",  # 31_humira_hyrimoz_gespoten_spuiten
    "A",  # 32_morgen_vanmorgen_morgenvroeg_ophalen
    "A",  # 33_jaar_vorig_jaren_leeftijd
    "M",  # 34_controle_ziekenhuis_check_periodieke
    "A",  # 35_vernomen_gebeld_helaas_extra
    "M",  # 36_dermatoloog_huid_dermatologie_gezicht
    "A",  # 37_volgende_mailen_hoeft_spuiten
    "A",  # 38_mee_hieraan_vreemd_valt
    "M",  # 39_antibiotica_kuur_antibioticakuur_bacterie
    "M",  # 40_operatie_chirurg_ingepland_gevolgen
    "A",  # 41_gelukkig_blij_gelukt_tevreden
    "A",  # 42_vakantie_vertrek_terug_reizen
    "M",  # 43_onderzoek_akkoord_melden_meedoen
    "A",  # 44_ibd_ibdcoach_sessie_vragen
    "M",  # 45_duurt_ruim_natuurlijk_lang
    "M",  # 46_koorts_temperatuur_verhoging_koude
    "A",  # 47_gaan_besproken_betekenen_houd
    "M",  # 48_mri_scan_dunne_uitslag
    "M",  # 49_zorgen_begin_ongerust_echt
    "M",  # 50_zwangerschap_zwanger_gynaecoloog_medicijngebruik
    "A",  # 51_fijne_feestdagen_jaarwisseling_gewenst
    "A",  # 52_plannen_inplannen_termijn_afspraak
    "A",  # 53_doorgeven_geven_volledig_oplossing
    "M",  # 54_keuze_opvlamming_bang_gal
    "A",  # 55_gegaan_mis_verkeerd_fout
    "A",  # 56_ontvangen_heden_moviprep_bevestiging
    "A",  # 57_zorginstelling_apotheek_ziekenhuis_naam
    "M",  # 58_coloscopie_endoscopie_darmonderzoek_verhuisd
    "A",  # 59_sorry_excuses_late_lastig
    "A",  # 60_zie_vlak_zien_verkeerde
    "A",  # 61_thuistesten_verwijzing_nieuwe_versturen
    "M",  # 62_dosering_combinatie_dosis_laag
    "A",  # 63_resultaten_hiervan_resultaat_vernemen
    "A",  # 64_bedoeling_teveel_doorgegeven_vergoed
    "A",  # 65_wilde_reden_dringend_denkt
    "M",  # 66_veranderd_infusen_helft_interval
    "A",  # 67_app_downloaden_telefoon_ondersteund
    "A",  # 68_voorraad_doosje_doos_stuks
    "A",  # 69_bespreken_spreken_persoonlijk_praten
    "A",  # 70_beste_hey_betreft_aandoen
    "M",  # 71_spuit_doorsturen_metamucil_injecties
    "M",  # 72_gewicht_kilo_afgevallen_weeg
    "M",  # 73_lactose_gestopt_beangstigend_gedaald
    "A",  # 74_nachten_duidelijk_kort_bestaat
    "A",  # 75_foto_bijgevoegd_bestand_bijgaand
    "A",  # 76_probleem_toekomst_echt_problemen
    "M",  # 77_stress_invloed_druk_rol
    "A",  # 78_zekerheid_navragen_vragenlijsten_checken
    "M",  # 79_pillen_systeem_azathiopirine_heen
    "A",  # 80_gehoord_gehoor_heden_kapper
    "M",  # 81_ontsteking_galstenen_idd_aanvallen
    "A",  # 82_vergeten_herinnering_herinneren_geheugen
    "M",  # 83_gestart_entyvio_gevaccineerd_verhuizen
    "A",  # 84_vandaar_gevraagd_griep_stuur
    "M",  # 85_ophalen_gang_fistel_bezig
    "M",  # 86_maagonderzoek_kloppen_vervolgafspraak_begrepen
    "A",  # 87_druppels_has_are_kale
    "M",  # 88_rechts_linker_steken_inspanning
    "A",  # 89_hiervoor_situatie_ten_eraan
    "A",  # 90_voldoende_hopende_hiermee_hoop
    "A",  # 91_gaten_houden_hou_gehouden
    "A",  # 92_helpen_kastje_hiermee_muur
    "A",  # 93_optie_alternatief_momenten_mogelijkheden
    "A",  # 94_vermelden_originele_hoor_biologische
    "A",  # 95_aanhouden_verstandig_rest_vervolg
    "A",  # 96_bekeken_lukken_betekend_geeft
    "M",  # 97_biologicals_biological_bureau_wratten
    "M",  # 98_risico_risicogroep_val_loop
    "M",  # 99_urine_kweek_gecontroleerd_urinekweek
    "A",  # 100_fax_faxnummer_vriendelijk_tel
    "M",  # 101_energie_energielevel_moe_stukken
    "A",  # 102_annuleren_meewerken_uitsluitsel_tijdstip
    "A",  # 103_toestemming_toestemmingsformulier_geef_bijlage
    "A",  # 104_wekelijks_nou_volgen_geplande
    "A",  # 105_verzekering_aanvraag_gegevens_levensverzekering
    "A",  # 106_duren_verbetering_langs_werken
    "M",  # 107_werkt_aangeef_traject_darmkrampen
    "A",  # 108_wensen_allereerst_gezond_wens
    "M",  # 109_bijwerking_bijwerkingen_infliximab_innemen
    "A",  # 110_eerstvolgende_aanstaande_groot_terecht
    "M",  # 111_problemen_lange_hierdoor_persen
    "A",  # 112_fysiek_opeens_heden_straks
    "A",  # 113_team_ibd_binnenkort_matig
    "A",  # 114_gezondheidscheck_periodieke_invullen_vullen
    "A",  # 115_gelezen_laboratorium_afgesproken_buikgriep
    "M",  # 116_tandarts_tandvlees_kaakchirurg_mond
    "A",  # 117_betekent_vervolgstappen_stappen_the
    "M",  # 118_aangepast_baat_beclometason_tabletjes
    "A",  # 119_tegemoet_zie_reactie_graag
    "A",  # 120_doorgestuurd_picoprep_gestuurd_zorginstelling
    "A",  # 121_geboortedatum_geboorte_geb_geboren
]
len(labels)

123

In [230]:
topic_info[topic_info["Name"].str.contains("operatie")]

,Topic,Count,Name,Representation,Representative_Docs,Label
41,40,217,40_operatie_chirurg_ingepland_gevolgen,"[operatie, chirurg, ingepland, gevolgen, vedoluzimab, chirurgie, verplaatsen, verergerd, afspraak, jaarlijkse]","[Graag zou ik nu weten wat dat voor een operatie zou zijn., Nu hoor ik net dat mijn operatie pas over 3 weken., of afgelopen maandag mijn operatie gehad.]",M


In [231]:
topic_info["Label"] = labels
topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs,Label
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, onderwerpen, genegeerd, docter, dexamfetamine, glucose, hogen]","[Of iets in die trant., Wat is voor mij het verstandigst., Of iets in die trant.]",A
1,0,4806,0_bloed_prikken_ontlasting_test,"[bloed, prikken, ontlasting, test, laten, slijm, bloedprikken, ingeleverd, bloeduitslagen, geprikt]","[Ik heb echter nog geen bloed laten prikken., Ik heb overigens gisterenochtend bloed laten prikken., m.v.g. Ben op in kerkrade geweest voor bloed te laten prikken.]",A
2,1,3651,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee...",M
3,2,3416,2_afspraak_telefonisch_contact_bellen,"[afspraak, telefonisch, contact, bellen, proberen, bereikbaar, vandaag, poli, gepland, opnemen]","[Kan een afspraak via u, of neem ik hier zelf contact over op?, 22 november heb ik telefonisch contact gehad met ., Anders moeten we nog even telefonisch contact op nemen]",A
4,3,3130,3_bedankt_dank_reactie_snelle,"[bedankt, dank, reactie, snelle, alvast, bericht, antwoord, dankjewel, fijn, weekend]","[Dank u voor de snelle reactie, Alvast dank voor jullie reactie., Bedankt voor uw snelle reactie.dat is heel fijn.]",A


In [232]:
doc_info = topic_model.get_document_info(sentences)
doc_info["Sentence_ID"] = doc_info.index + 1 
doc_info.head()

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document,Sentence_ID
0,Ik ben 2 weken geleden met spoed opgenomen in het in met verdenking van hartklachten.,45,45_duurt_ruim_natuurlijk_lang,"[duurt, ruim, natuurlijk, lang, hoesten, duurde, moedeloos, toegenomen, vroeger, zorg]","[Maakt niks uit als het nog even duurt., Weet U hoe lang het duurt eer uitslag binnen is ?, Deze laatste duurt ruim 3 uur.]",duurt - ruim - natuurlijk - lang - hoesten - duurde - moedeloos - toegenomen - vroeger - zorg,False,1
1,"Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.",1,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee...",pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,False,2
2,Het begon 1 uur na het avondeten.,1,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee...",pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,False,3
3,"Ik had al de hele dag migraine, had dus ook weinig gegeten.",1,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee...",pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,False,4
4,"Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.",1,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee...",pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,False,5


In [233]:
annotated_df_new = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/new_annotated_df_lg.xlsx")
annotated_df_new.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence,Manual_Category
0,3,Ik ben 2 weken geleden met spoed opgenomen in het [PERSOON] in [LOCATIE] met verdenking van hartklachten.,Ik ben 2 weken geleden met spoed opgenomen in het in met verdenking van hartklachten.,1,I was rushed into the [PRESSION] two weeks ago in [LOCATION] with suspicion of heart problems.,M
1,3,"Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.","Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.",2,"I was acutely under a painful pressure on the chest, shoulder blades, radiating to the arms.",M
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.,M
3,3,"Ik had al de hele dag migraine, had dus ook weinig gegeten.","Ik had al de hele dag migraine, had dus ook weinig gegeten.",4,"I had migraines all day, so I hadn't eaten much.",M
4,3,"Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.","Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.",5,"I got very nauseous, vomiting, dizzy and rejuvenating, and terrible pain.",M


In [234]:
# now we need to move the labels of the topic_info to doc_info
# Now perform the merge 
annotated_doc_info = pd.merge( # merging the small annotated set with the document info
    annotated_df_new,
    doc_info[['Sentence_ID', 'Topic', 'Top_n_words']], # notice that we only choose these 3 columns of the doc info df
    on='Sentence_ID',
    how='left'
)

annotated_doc_info.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence,Manual_Category,Topic,Top_n_words
0,3,Ik ben 2 weken geleden met spoed opgenomen in het [PERSOON] in [LOCATIE] met verdenking van hartklachten.,Ik ben 2 weken geleden met spoed opgenomen in het in met verdenking van hartklachten.,1,I was rushed into the [PRESSION] two weeks ago in [LOCATION] with suspicion of heart problems.,M,45,duurt - ruim - natuurlijk - lang - hoesten - duurde - moedeloos - toegenomen - vroeger - zorg
1,3,"Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.","Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.",2,"I was acutely under a painful pressure on the chest, shoulder blades, radiating to the arms.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.,M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree
3,3,"Ik had al de hele dag migraine, had dus ook weinig gegeten.","Ik had al de hele dag migraine, had dus ook weinig gegeten.",4,"I had migraines all day, so I hadn't eaten much.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree
4,3,"Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.","Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.",5,"I got very nauseous, vomiting, dizzy and rejuvenating, and terrible pain.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree


In [235]:
# now merge the above dataframe with the labeled topic info frame
annotated_doc_info = pd.merge(
    annotated_doc_info,
    topic_info[['Topic', 'Label']],
    on='Topic',
    how='left'
)
annotated_doc_info.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence,Manual_Category,Topic,Top_n_words,Label
0,3,Ik ben 2 weken geleden met spoed opgenomen in het [PERSOON] in [LOCATIE] met verdenking van hartklachten.,Ik ben 2 weken geleden met spoed opgenomen in het in met verdenking van hartklachten.,1,I was rushed into the [PRESSION] two weeks ago in [LOCATION] with suspicion of heart problems.,M,45,duurt - ruim - natuurlijk - lang - hoesten - duurde - moedeloos - toegenomen - vroeger - zorg,M
1,3,"Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.","Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.",2,"I was acutely under a painful pressure on the chest, shoulder blades, radiating to the arms.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.,M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
3,3,"Ik had al de hele dag migraine, had dus ook weinig gegeten.","Ik had al de hele dag migraine, had dus ook weinig gegeten.",4,"I had migraines all day, so I hadn't eaten much.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
4,3,"Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.","Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.",5,"I got very nauseous, vomiting, dizzy and rejuvenating, and terrible pain.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M


In [237]:
annotated_doc_info[500:600]

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence,Manual_Category,Topic,Top_n_words,Label
500,323,Ik las trouwens dat ik door de contrast vloeistof de moedermelk die dag moet weggooien.,Ik las trouwens dat ik door de contrast vloeistof de moedermelk die dag moet weggooien.,510,"By the way, I read that because of the contrast liquid I must throw away breast milk that day.",M,83,gestart - entyvio - gevaccineerd - verhuizen - afspraken - trouwens - gemaakt - uitgerekend - gepaard - kwam,M
501,323,Is dat alleen die eerste dag of ook de dag erna nog?,Is dat alleen die eerste dag of ook de dag erna nog?,511,Is that just the first day or the next day?,M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
502,325,Afgelopen week een voorhoofdsholte ontsteking gehad.,Afgelopen week een voorhoofdsholte ontsteking gehad.,512,"Last week, he had a sinusitis.",M,0,bloed - prikken - ontlasting - test - laten - slijm - bloedprikken - ingeleverd - bloeduitslagen - geprikt,A
503,325,Sindsdien alleen maar waterige ontlasting ik voel me heel slap en ben ontzettend veel afgevallen.,Sindsdien alleen maar waterige ontlasting ik voel me heel slap en ben ontzettend veel afgevallen.,513,Since then I feel very weak and have lost a lot of weight.,M,10,voel - gevoel - voelt - voelen - moe - gevoelig - voelde - lekker - goed - slecht,M
504,325,Graag wat advies wat ik nu het beste kan doen.,Graag wat advies wat ik nu het beste kan doen.,514,I'd like some advice that I can do best right now.,M,70,beste - hey - betreft - aandoen - voorbereiden - plichten - picoprepp - aangetoond - onafhankelijkheid - middel,A
...,...,...,...,...,...,...,...,...,...
595,400,Verder heb ik vorige week op een nacht veel krampen in mijn maag en rug gehad.,Verder heb ik vorige week op een nacht veel krampen in mijn maag en rug gehad.,605,"Furthermore, last week I had a lot of cramps in my stomach and back.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
596,400,Hier kon ik niet van slapen.,er kon ik niet van slapen.,606,This didn't help me sleep.,M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
597,400,"Dit heb ik eerder nog niet zo lang geleden nog een keer precies hetzelfde gehad, maar toen begon het s'ochtends en had ik het de hele dag, avond en nacht.","Dit heb ik eerder nog niet zo lang geleden nog een keer precies hetzelfde gehad, maar toen begon het s'ochtends en had ik het de hele dag, avond en nacht.",607,"I had the exact same thing not so long ago before, but then it started in the morning and I had it all day, night and night.",M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M
598,400,Waarschijnlijk had ik de dag voordat het begon bij beide keren te pittig of te vettig gegeten.,Waarschijnlijk had ik de dag voordat het begon bij beide keren te pittig of te vettig gegeten.,608,Probably I had eaten too spicy or too greasy the day before it started at both times.,M,1,pijn - buikpijn - toilet - last - keer - buik - klachten - normaal - soms - diarree,M


In [236]:
accuracy = (annotated_doc_info["Label"] == annotated_doc_info["Manual_Category"]).mean()

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 69.22%


In [239]:
# Accuracy for "M"
m_accuracy = (annotated_doc_info[annotated_doc_info["Label"] == "M"]["Manual_Category"] == "M").mean()

# Accuracy for "A"
a_accuracy = (annotated_doc_info[annotated_doc_info["Label"] == "A"]["Manual_Category"] == "A").mean()

print(f"Accuracy for M: {m_accuracy:.2%}")
print(f"Accuracy for A: {a_accuracy:.2%}")


Accuracy for M: 61.04%
Accuracy for A: 74.38%


In [240]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs,Label
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, onderwerpen, genegeerd, docter, dexamfetamine, glucose, hogen]","[Of iets in die trant., Wat is voor mij het verstandigst., Of iets in die trant.]",A
1,0,4806,0_bloed_prikken_ontlasting_test,"[bloed, prikken, ontlasting, test, laten, slijm, bloedprikken, ingeleverd, bloeduitslagen, geprikt]","[Ik heb echter nog geen bloed laten prikken., Ik heb overigens gisterenochtend bloed laten prikken., m.v.g. Ben op in kerkrade geweest voor bloed te laten prikken.]",A
2,1,3651,1_pijn_buikpijn_toilet_last,"[pijn, buikpijn, toilet, last, keer, buik, klachten, normaal, soms, diarree]","[maar ik heb tot nu toe nog steeds last van buikpijn en pijn aan mijn maag als ik het goed heb .. nu is de vraag is dat normaal of niet ., Ik heb veel pijn in mijn buik., Wel wat diarree, maar gee...",M
3,2,3416,2_afspraak_telefonisch_contact_bellen,"[afspraak, telefonisch, contact, bellen, proberen, bereikbaar, vandaag, poli, gepland, opnemen]","[Kan een afspraak via u, of neem ik hier zelf contact over op?, 22 november heb ik telefonisch contact gehad met ., Anders moeten we nog even telefonisch contact op nemen]",A
4,3,3130,3_bedankt_dank_reactie_snelle,"[bedankt, dank, reactie, snelle, alvast, bericht, antwoord, dankjewel, fijn, weekend]","[Dank u voor de snelle reactie, Alvast dank voor jullie reactie., Bedankt voor uw snelle reactie.dat is heel fijn.]",A
...,...,...,...,...,...,...
118,117,42,117_betekent_vervolgstappen_stappen_the,"[betekent, vervolgstappen, stappen, the, vervolg, vervolgstap, leukocyten, rapportage, attach, far]","[Wat betekent dit nu?, Wat betekent dit?, Wat betekent dit?]",A
119,118,39,118_aangepast_baat_beclometason_tabletjes,"[aangepast, baat, beclometason, tabletjes, middel, nachtrust, weerstand, beviel, concluderen, echinacea]","[Zou dit ook nog aangepast kunnen worden?, De gezondheidspremie wordt aangepast., Ik heb hier toch veel meer baat bij dan bij de Beclometason 3x per week en nu gaat het goed met de klachten.]",M
120,119,37,119_tegemoet_zie_reactie_graag,"[tegemoet, zie, reactie, graag, hieronder, kliniek, papiertje, premievrijstelling, afsprakenbrief, pensioenfonds]","[Ik zie graag jullie reactie tegemoet., Graag zie ik u reactie tegemoet., Ik zie je reactie graag tegemoet.]",A
121,120,36,120_doorgestuurd_picoprep_gestuurd_zorginstelling,"[doorgestuurd, picoprep, gestuurd, zorginstelling, doorgezet, derrein, recept, apotheek, gram, hartbewaking]","[Dit mag doorgestuurd worden naar de [APOTHEEK] in ., Dit mag doorgestuurd worden naar de [APOTHEEK] in ., Het recept van de picoprep zou zijn doorgestuurd naar [APOTHEEK]derrein .]",A


In [245]:

# Example DataFrame (replace with your actual data)

df = topic_info[topic_info["Label"] == "M"]
df = df[["Count", "Name", "Label"]][1:]
df.head()

,Count,Name,Label
5,2298,4_medicatie_gebruik_tabletten_medicijnen,M
6,1559,5_huisarts_arts_dokter_mdl,M
8,1074,7_bekend_calprotectine_uitslag_waarde,M
11,558,10_voel_gevoel_voelt_voelen,M
12,553,11_corona_vaccinatie_vaccin_positief,M


In [248]:
import pandas as pd
import matplotlib.pyplot as plt

# Example DataFrame (replace with your actual data)


# Pivot the data so that we can have 'A' and 'M' in separate columns for each label
pivot_df = df.pivot(index='Name', columns='Label', values='Count').fillna(0)

# Sort by the 'Total' column in descending order (highest to lowest)
pivot_df = pivot_df.sort_values(by='Total')

# Create a figure and axes
fig, ax = plt.subplots(figsize=(10, 6))

# Plot Category M on the left side of the plot (negative values for left-side bars)
ax.barh(pivot_df.index, pivot_df['M'], color='#A8D5BA', label='Medical', height=0.6)  # Softer green

# Add labels and title
ax.set_xlabel('Count')
ax.set_title('Topic frequency')
ax.legend()

# Show the plot
plt.tight_layout()
plt.show()


KeyError: 'Total'

## Visualize data